In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
import pandas as pd
import json

# Load the data from JSON
with open('/content/drive/MyDrive/ASD Level/Metadata/data.json', 'r') as f:
    data_json = json.load(f)

# Convert the 'TCImages' part of the JSON to a DataFrame
df = pd.DataFrame(data_json['TSImages'])
df = df.sort_values('ParticipantID').reset_index(drop = True)
df

,File,ParticipantID,Age,Gender,CARS
0,TS126_01.png,1,7.0,M,32.5
1,TS127_01.png,1,7.0,M,32.5
2,TS128_01.png,1,7.0,M,32.5
3,TS129_01.png,1,7.0,M,32.5
4,TS134_01.png,1,7.0,M,32.5
...,...,...,...,...,...
214,TS218_08.png,8,12.3,F,34.5
215,TS120_09.png,9,2.7,M,27.0
216,TS195_09.png,9,2.7,M,27.0
217,TS118_09.png,9,2.7,M,27.0


In [39]:
# Add the 'Label' column based on the CARS value
df['Label'] = df['CARS'].apply(lambda x: 0 if x < 30 else 1)

df.head()

,File,ParticipantID,Age,Gender,CARS,Label
0,TS126_01.png,1,7.0,M,32.5,1
1,TS127_01.png,1,7.0,M,32.5,1
2,TS128_01.png,1,7.0,M,32.5,1
3,TS129_01.png,1,7.0,M,32.5,1
4,TS134_01.png,1,7.0,M,32.5,1


In [45]:
# Perform train-test split (30% test data)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Label'])

print("Train set size:", len(train_df))
print("Test set size:", len(test_df))

Train set size: 175
Test set size: 44


In [46]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class TSImagesDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.dataframe.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = self.dataframe.iloc[idx, -1]  # Assuming 'Label' is the last column

        if self.transform:
            image = self.transform(image)

        return image, label

# Define the image directory and transformations
img_dir = '/content/drive/MyDrive/ASD Level/Images/TSImages'
transform = transforms.Compose([
    transforms.Resize((100, 100)),  # Resize images to 100x100 pixels
    transforms.ToTensor()  # Convert images to PyTorch tensors
])

# Create the training and testing datasets
train_dataset = TSImagesDataset(dataframe=train_df, img_dir=img_dir, transform=transform)
test_dataset = TSImagesDataset(dataframe=test_df, img_dir=img_dir, transform=transform)

# Create DataLoaders for training and testing sets
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Example: Iterate over the training DataLoader
for images, labels in train_dataloader:
    print("Train batch:")
    print(images.shape)  # Should print torch.Size([32, 3, 100, 100]) for a batch of 32 images
    print(labels)        # Labels for the batch
    break  # Just to test a single batch

# Example: Iterate over the testing DataLoader
for images, labels in test_dataloader:
    print("Test batch:")
    print(images.shape)  # Should print torch.Size([32, 3, 100, 100]) for a batch of 32 images
    print(labels)        # Labels for the batch
    break  # Just to test a single batch


Train batch:
torch.Size([32, 3, 100, 100])
tensor([0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0,
        1, 0, 0, 1, 1, 1, 1, 0])
Test batch:
torch.Size([32, 3, 100, 100])
tensor([0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 0])


In [61]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Load the pretrained ResNet18 model
model = models.resnet18(pretrained=True)

# Freeze the early layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer to match the number of classes in your task
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Assuming 2 classes (label 0 and label 1)

# Move the model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training the model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

print("Finished Training")


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/100], Loss: 0.8245
Epoch [2/100], Loss: 0.7382
Epoch [3/100], Loss: 0.7071
Epoch [4/100], Loss: 0.6260
Epoch [5/100], Loss: 0.6216
Epoch [6/100], Loss: 0.5817
Epoch [7/100], Loss: 0.5549
Epoch [8/100], Loss: 0.5304
Epoch [9/100], Loss: 0.4946
Epoch [10/100], Loss: 0.4807
Epoch [11/100], Loss: 0.4875
Epoch [12/100], Loss: 0.4656
Epoch [13/100], Loss: 0.4437
Epoch [14/100], Loss: 0.4646
Epoch [15/100], Loss: 0.4710
Epoch [16/100], Loss: 0.4609
Epoch [17/100], Loss: 0.4721
Epoch [18/100], Loss: 0.4238
Epoch [19/100], Loss: 0.3811
Epoch [20/100], Loss: 0.4323
Epoch [21/100], Loss: 0.3867
Epoch [22/100], Loss: 0.3680
Epoch [23/100], Loss: 0.3621
Epoch [24/100], Loss: 0.3436
Epoch [25/100], Loss: 0.3751
Epoch [26/100], Loss: 0.3647
Epoch [27/100], Loss: 0.3425
Epoch [28/100], Loss: 0.3863
Epoch [29/100], Loss: 0.3464
Epoch [30/100], Loss: 0.3543
Epoch [31/100], Loss: 0.3343
Epoch [32/100], Loss: 0.3343
Epoch [33/100], Loss: 0.3189
Epoch [34/100], Loss: 0.3086
Epoch [35/100], Loss: 0

In [62]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the test set
model.eval()  # Set model to evaluation mode
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(inputs)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Store labels and predictions
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")



Test Accuracy: 0.6818
Test Precision: 0.6389
Test Recall: 0.9583
Test F1 Score: 0.7667


In [63]:
pd.DataFrame([all_labels, all_preds])

,0,1,2,3,4,5,6,7,8,9,...,34,35,36,37,38,39,40,41,42,43
0,0,1,1,0,0,0,1,1,0,1,...,0,0,1,1,1,0,0,1,0,0
1,0,1,1,1,1,1,1,1,0,1,...,0,1,1,1,1,1,1,1,1,0
